# Notebook 01: Data Cleaning, Audit & Validation Pipeline
## Project: Obesity & Lifestyle Exploratory Data Analysis
**Course:** F.Y. BCA Semester I (Div A) | **Group:** 5  
**Submission Date:** 5th October 2026

---

### Objective
This notebook performs an exhaustive quality audit and data validation workflow on the UCI Obesity & Lifestyle dataset:
1. **Raw Data Ingestion:** Load and inspect the 2,111-record UCI benchmark (`data/raw/ObesityDataSet_raw_and_data_sinthetic.csv`).
2. **Leakage Prevention Audit:** Validate why biometrics (`Height`, `Weight`) are excluded in behavioral/lifestyle modeling to avoid circular BMI calculation leakage.
3. **Cleaned Cohort Ingestion:** Load the 515-record audited survey dataset (`data/cleaned/ObesityDataSet_cleaned.csv`).
4. **Data Hygiene Checks:** Missing value audit, data type integrity, categorical level consistency, and numerical range verification.
5. **Duplicate & Synthetic Sampling Detection:** Identify and quantify the 29 duplicate records (including the 15-record consecutive identical synthetic cluster).
6. **Automated Assertion Suite:** Automated test assertions validating clean data status before exploratory analysis.


In [ ]:
import pandas as pd
import numpy as np
import os

print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)


## 1. Raw Dataset Ingestion & Structure Audit

In [ ]:
raw_path = os.path.join("..", "data", "raw", "ObesityDataSet_raw_and_data_sinthetic.csv")
if not os.path.exists(raw_path):
    raw_path = os.path.join("data", "raw", "ObesityDataSet_raw_and_data_sinthetic.csv")

raw_df = pd.read_csv(raw_path)
print(f"[RAW DATASET] Shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
raw_df.head()


### Feature Inventory of Raw Dataset
Notice that the raw dataset contains `Height` and `Weight`. Since the clinical target `NObeyesdad` is calculated directly via BMI:
$$\text{BMI} = \frac{\text{Weight (kg)}}{\text{Height (m)}^2}$$
Keeping `Height` and `Weight` creates deterministic circular leakage. In our behavioral analysis and machine learning preparation, our objective is to evaluate lifestyle habits, genetics, and demographics as predictors, making the removal of raw biometrics necessary.


In [ ]:
raw_cols = list(raw_df.columns)
print("Raw columns (17 total):")
for i, col in enumerate(raw_cols, 1):
    print(f"  {i:02d}. {col}")


## 2. Ingesting Cleaned Empirical Cohort

In [ ]:
cleaned_path = os.path.join("..", "data", "cleaned", "ObesityDataSet_cleaned.csv")
if not os.path.exists(cleaned_path):
    cleaned_path = os.path.join("data", "cleaned", "ObesityDataSet_cleaned.csv")

df = pd.read_csv(cleaned_path)
print(f"[CLEANED DATASET] Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.info()


## 3. Data Hygiene & Missing Value Audit

In [ ]:
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()

print("--- Missing Value Audit ---")
print(f"Total Missing / Null Values: {total_nulls}")
print(null_counts)

# Assertion test
assert total_nulls == 0, f"Validation Failed: Found {total_nulls} missing values."
print("\n[PASSED] 100% Completeness Verified: Exactly 0 missing values across all columns.")


## 4. Categorical Consistency & Bounds Verification

In [ ]:
# Define expected domain taxonomies
expected_domains = {
    'Gender': {'female', 'male'},
    'family_history_with_overweight': {'yes', 'no'},
    'FAVC': {'yes', 'no'},
    'CAEC': {'no', 'sometimes', 'frequently', 'always'},
    'SMOKE': {'yes', 'no'},
    'SCC': {'yes', 'no'},
    'CALC': {'no', 'sometimes', 'frequently', 'always'},
    'MTRANS': {'automobile', 'bike', 'motorbike', 'public_transportation', 'walking'},
    'NObeyesdad': {
        'insufficient_weight', 'normal_weight', 'overweight_level_i',
        'overweight_level_ii', 'obesity_type_i', 'obesity_type_ii', 'obesity_type_iii'
    }
}

for col, allowed in expected_domains.items():
    unique_vals = set(df[col].unique())
    unexpected = unique_vals - allowed
    assert len(unexpected) == 0, f"Error in {col}: Unexpected categories {unexpected}"
    print(f"[PASSED] Column '{col}' valid. Unique values: {sorted(list(unique_vals))}")


## 5. Numerical Range Verification

In [ ]:
numeric_ranges = {
    'Age': (14, 100),
    'FCVC': (1, 3),
    'NCP': (1, 4),
    'CH2O': (1, 3),
    'FAF': (0, 3),
    'TUE': (0, 2)
}

for col, (min_val, max_val) in numeric_ranges.items():
    observed_min = df[col].min()
    observed_max = df[col].max()
    assert observed_min >= min_val and observed_max <= max_val, f"Range error in {col}"
    print(f"[PASSED] '{col}': Min = {observed_min}, Max = {observed_max} (Expected: [{min_val}, {max_val}])")


## 6. Duplicate Records & Synthetic Cluster Detection

In [ ]:
num_duplicates = df.duplicated().sum()
pct_duplicates = (num_duplicates / len(df)) * 100
print(f"Total Duplicate Rows: {num_duplicates} ({pct_duplicates:.2f}% of cohort)")

# Display duplicate records grouped by frequency
duplicate_groups = df[df.duplicated(keep=False)].groupby(list(df.columns)).size().reset_index(name='count')
print("\nDuplicate Clusters identified:")
duplicate_groups.sort_values(by='count', ascending=False)


### Audit Takeaway: The Synthetic Balancing Artifact
Notice cluster #1:
- Male, 21y, Family Hist: No, FAVC: Yes, FCVC: 2, NCP: 1, CAEC: No, SMOKE: No, CH2O: 3, SCC: No, FAF: 1, TUE: 0, CALC: Sometimes, MTRANS: Public Transit, Target: Overweight Level I
- This exact vector appears **15 consecutive times** at the end of the survey data.
- **Data Engineering Action:** When splitting into Train / Test sets for downstream machine learning, standard random k-fold will cause data leakage. A stratified group or deduplicated validation split is required.


## 7. Data Validation Summary & Export Verification

In [ ]:
validation_summary = {
    'Total Rows Audited': len(df),
    'Total Columns': df.shape[1],
    'Missing Values': int(total_nulls),
    'Duplicate Rows': int(num_duplicates),
    'Target Classes Present': df['NObeyesdad'].nunique(),
    'Data Status': 'PASSED - Ready for EDA'
}

summary_df = pd.DataFrame([validation_summary])
summary_df
